# Laboratorio 6 - Amostragem do Sinal Senoidal

Este notebook implementa os exercicios do Laboratorio 6 sobre amostragem de sinais senoidais em tempo discreto.

Um sinal senoidal continuo pode ser escrito como:

$$f(t) = \sin(2\pi f t)$$

Quando amostramos esse sinal com periodo de amostragem $T_s$, obtemos:

$$x[n] = \sin(2\pi f T_s n)$$

A frequencia de amostragem e:

$$F_s = \frac{1}{T_s}$$


In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np


def sampled_sine(freq_hz: float, sampling_period: float, duration: float) -> tuple[float, np.ndarray, np.ndarray]:
    fs = 1.0 / sampling_period
    t = np.arange(0.0, duration, sampling_period)
    x = np.sin(2 * np.pi * freq_hz * t)
    return fs, t, x


def continuous_sine(freq_hz: float, duration: float, points: int = 4000) -> tuple[np.ndarray, np.ndarray]:
    t = np.linspace(0.0, duration, points, endpoint=False)
    x = np.sin(2 * np.pi * freq_hz * t)
    return t, x


def plot_sampling_cases(freq_hz: float, sampling_periods: list[float], duration: float, title: str) -> None:
    t_ref, x_ref = continuous_sine(freq_hz, duration)

    fig, axes = plt.subplots(len(sampling_periods), 1, figsize=(11, 3.2 * len(sampling_periods)), sharex=True)
    axes = np.atleast_1d(axes)

    for ax, ts in zip(axes, sampling_periods):
        fs, t_samples, x_samples = sampled_sine(freq_hz, ts, duration)

        ax.plot(t_ref, x_ref, color="0.65", linewidth=1.2, label="Referencia continua")
        ax.stem(t_samples, x_samples, linefmt="C0-", markerfmt="C0o", basefmt=" ", label="Amostras")
        ax.set_title(f"Ts = {ts:.5f} s | Fs = {fs:.1f} Hz | {len(t_samples)} amostras")
        ax.set_ylabel("Amplitude")
        ax.grid(True)
        ax.legend(loc="upper right")

    axes[-1].set_xlabel("Tempo (s)")
    fig.suptitle(title, y=1.01)
    plt.tight_layout()
    plt.show()


## 1) Senoide de 1 Hz com diferentes periodos de amostragem

Neste item, o sinal continuo e:

$$f(t) = \sin(2\pi \cdot 1 \cdot t)$$

Foram usados os periodos pedidos:

- $T_s = 1/100$ s, logo $F_s = 100$ Hz
- $T_s = 1/10$ s, logo $F_s = 10$ Hz
- $T_s = 1/2$ s, logo $F_s = 2$ Hz


In [ ]:
freq_sinal = 1.0
sampling_periods_1hz = [1 / 100, 1 / 10, 1 / 2]

plot_sampling_cases(
    freq_hz=freq_sinal,
    sampling_periods=sampling_periods_1hz,
    duration=2.0,
    title="Amostragem de uma senoide de 1 Hz",
)


## Analise do item 1

Com $T_s = 1/100$ s, a frequencia de amostragem e $100$ Hz. Isso gera muitas amostras por ciclo da senoide de 1 Hz, entao o formato do sinal fica bem representado.

Com $T_s = 1/10$ s, a frequencia de amostragem e $10$ Hz. Ainda ha 10 amostras por ciclo, entao o sinal continua reconhecivel, mas com menos detalhes.

Com $T_s = 1/2$ s, a frequencia de amostragem e $2$ Hz. Esse caso fica exatamente no limite de Nyquist para um sinal de 1 Hz. Como a senoide foi amostrada nos instantes em que cruza zero, as amostras ficam praticamente todas nulas. Assim, mesmo estando no limite teorico, a representacao fica ruim porque os pontos escolhidos nao mostram a forma da senoide.

A diferenca entre os periodos de amostragem e que, quanto menor $T_s$, maior $F_s$ e maior a quantidade de amostras por segundo. Quanto maior $T_s$, menor $F_s$ e mais pobre fica a representacao do sinal.

## 2) Senoide de 30 Hz e teorema da amostragem

Agora o sinal tem frequencia:

$$f = 30\text{ Hz}$$

Pelo teorema da amostragem, para representar corretamente esse sinal, a frequencia de amostragem precisa ser maior que o dobro da frequencia do sinal:

$$F_s > 2f$$

Logo:

$$F_s > 60\text{ Hz}$$

ou, em termos de periodo:

$$T_s < \frac{1}{60}\text{ s}$$

Foram escolhidos dois casos corretos e um caso com aliasing:

- Correto: $T_s = 1/200$ s, logo $F_s = 200$ Hz
- Correto: $T_s = 1/100$ s, logo $F_s = 100$ Hz
- Aliasing: $T_s = 1/40$ s, logo $F_s = 40$ Hz


In [ ]:
freq_sinal = 30.0
sampling_periods_30hz = [1 / 200, 1 / 100, 1 / 40]

plot_sampling_cases(
    freq_hz=freq_sinal,
    sampling_periods=sampling_periods_30hz,
    duration=0.2,
    title="Amostragem de uma senoide de 30 Hz",
)


In [ ]:
def alias_frequency(freq_hz: float, fs_hz: float) -> float:
    return abs(((freq_hz + fs_hz / 2) % fs_hz) - fs_hz / 2)


for ts in sampling_periods_30hz:
    fs = 1 / ts
    alias = alias_frequency(freq_sinal, fs)
    status = "correto" if fs > 2 * freq_sinal else "aliasing"
    print(f"Ts = {ts:.5f} s | Fs = {fs:.1f} Hz | status: {status} | frequencia aparente: {alias:.1f} Hz")


## Analise do item 2

Para o sinal de 30 Hz, os casos com $F_s = 200$ Hz e $F_s = 100$ Hz respeitam o teorema da amostragem, pois ambos sao maiores que $60$ Hz. Por isso, a senoide e representada corretamente.

No caso com $F_s = 40$ Hz, a frequencia de amostragem e menor que $2f = 60$ Hz. Portanto ocorre aliasing. O sinal original de 30 Hz passa a ser visto nas amostras como uma senoide de frequencia aparente menor. Para $F_s = 40$ Hz, a frequencia aparente e:

$$f_{alias} = |30 - 40| = 10\text{ Hz}$$

Isso mostra que, quando a frequencia de amostragem e insuficiente, as amostras podem representar uma frequencia falsa, diferente da frequencia real do sinal continuo.